# 비지도학습: 클러스터링 + 차원축소(PCA) — Digit Recognizer — Colab

라벨 없이 데이터를 **군집화**하고 **차원을 축소**하는 비지도 기본 예제입니다.

- 핵심 흐름: **PCA** 로 차원축소·시각화 → **KMeans** 군집 → 군집-실제라벨 일치도(ARI/NMI) 평가.
- IOAI 2025 **Antique**(`SpectralClustering`), 2024 **Lost in Hyperspace**(`PCA`) 의 기반 기법입니다.
- [Digit Recognizer](https://www.kaggle.com/c/digit-recognizer) 사용. 토큰이 **노트북에 내장**되어 바로 실행됩니다.

> ⚙️ CPU 로 충분. ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부 공유 금지.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "scikit-learn", "numpy", "pandas", "matplotlib"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Digit Recognizer 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 로드

비지도이므로 라벨(`label`)은 **평가용으로만** 쓰고 학습엔 사용하지 않습니다.

In [ ]:
import numpy as np, pandas as pd
df = pd.read_csv(os.path.join(WORK_DIR, "train.csv")).sample(10000, random_state=0)
X = df.drop("label", axis=1).values.astype("float32")/255.0
y = df["label"].values  # 평가용
print("X:", X.shape, "(라벨은 평가에만 사용)")

## 3. PCA 차원축소 & 설명분산

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

pca50 = PCA(n_components=50, random_state=0).fit(X)
X50 = pca50.transform(X)
print("누적 설명분산(50차원):", round(pca50.explained_variance_ratio_.sum(), 3))

p2 = PCA(n_components=2, random_state=0).fit_transform(X)
plt.figure(figsize=(6,5))
plt.scatter(p2[:,0], p2[:,1], c=y, cmap="tab10", s=5, alpha=.5)
plt.colorbar(label="true digit"); plt.title("PCA 2D of MNIST (color=true label)"); plt.show()

## 4. KMeans 군집화 & 평가

10개 군집으로 KMeans 를 수행하고, 실제 숫자 라벨과의 일치도를 ARI/NMI 로 측정합니다(라벨은 평가에만).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.stats import mode

km = KMeans(n_clusters=10, n_init=10, random_state=0).fit(X50)
clusters = km.labels_

ari = adjusted_rand_score(y, clusters)
nmi = normalized_mutual_info_score(y, clusters)

# 군집→다수결 라벨 매핑으로 "군집 정확도" 계산
mapped = np.zeros_like(clusters)
for c in range(10):
    m = clusters == c
    if m.sum(): mapped[m] = mode(y[m], keepdims=True).mode[0]
acc = (mapped == y).mean()
print(f"ARI {ari:.3f} | NMI {nmi:.3f} | 군집 다수결 정확도 {acc:.3f}")

## 5. 군집 중심 시각화

각 KMeans 군집 중심(50→784 역변환)을 이미지로 보면 숫자 형태가 드러납니다.

In [ ]:
centers = pca50.inverse_transform(km.cluster_centers_).reshape(10, 28, 28)
fig, ax = plt.subplots(1, 10, figsize=(13, 1.6))
for i in range(10):
    ax[i].imshow(centers[i], cmap="gray"); ax[i].axis("off")
plt.suptitle("KMeans cluster centers"); plt.show()

## 🏁 제출 안내

이 노트북은 **비지도학습(클러스터링+차원축소)** 연습용 데모입니다(제출 리더보드 없음).

- 데이터 출처: **[Digit Recognizer](https://www.kaggle.com/c/digit-recognizer)**

> IOAI 2025 *Antique*(SpectralClustering), 2024 *Lost in Hyperspace*(PCA) 의 기반 기법입니다.